# Lab 1.1 &mdash; From a Stateless Call to an Agent Loop

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 1 &middot; Module 1 &mdash; Agents vs. Multi-Agent Systems**

### What you'll do
- Carry state the way LangChain does it &mdash; as a list of message objects you resend
- Let the model choose a tool for real, with `bind_tools` and `tool_calls`
- Close the loop by feeding results back as `ToolMessage`, and make it stop
- Then replace the whole thing with `create_agent` + a checkpointer, and compare

> **How this lab works.** You write real LangChain and LangGraph code. Fill every `BLANK`,
> then run the **Self-check** cell under each section &mdash; those check the *objects you built*
> (a bound tool, a compiled graph, an emitted tool call), so they are deterministic and do not
> depend on the model. Cells marked **Run it for real** put your code in front of the sandbox
> model; that is the part worth watching. The score line is feedback, not a grade.

> **The thread.** All five Module 1 labs work one case: payment exceptions on a small
> synthetic ledger. What you build here is extended in every later lab.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-1-01")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nSelf-check: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens: 24.1s / 980 tokens with it on, 0.7s / 29 with it off, for the same answer. Off is
# the default here because you will make a lot of calls today. Pass think=True to see the
# difference for yourself -- and note that prompts written as an explicit ordered procedure
# survive thinking being off, while vague ones do not.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

def show_messages(messages, width: int = 88) -> None:
    """Print a message list the way a trace reads: type, content, and any tool calls."""
    for m in messages:
        kind = getattr(m, "type", "?")
        body = str(getattr(m, "content", "")).replace("\n", " ")[:width]
        calls = getattr(m, "tool_calls", None)
        line = f"  [{kind:9}] {body}"
        if calls:
            line += "  -> calls: " + ", ".join(f"{c['name']}({c['args']})" for c in calls)
        print(line)

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 1 labs: payment exceptions on a small ledger.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

## Concept

A model call is a **function**: messages in, message out, nothing retained. An **agent** is that
call placed inside a **loop**, where the model's output chooses the next action and the result is
fed back in as another message.

In LangChain that loop has a precise shape, and it is worth learning the names now because every
later module uses them:

| Object | What it is |
|---|---|
| `HumanMessage` / `AIMessage` / `SystemMessage` | the conversation, as data you own |
| `llm.bind_tools([...])` | a model that is allowed to answer with a **tool call** |
| `AIMessage.tool_calls` | the model's chosen action &mdash; structured, not parsed out of prose |
| `ToolMessage` | the result you hand back, tied to the call by `tool_call_id` |

Three things make the loop safe rather than merely clever: **state**, a **stop condition**, and
**loop detection**. The last two are what separate a demo from something you would run unattended.

## Section 1 &mdash; State is a list of messages you resend

The model has no memory, so *you* carry the conversation. `carry()` builds the message list for
the next call: a system message, every earlier turn, then the new human message.

These are real `langchain_core` objects, not tuples &mdash; every later lab, and LangGraph itself,
passes exactly this list around.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

SYSTEM = ("You are a payments operations analyst. Answer only from the data you are given. "
          "If you do not have the data, say so.")

def carry(history: list, user_msg: str) -> list:
    """Build the message list for the next call.

    history: earlier message objects, oldest first.
    Returns: [SystemMessage, *history, HumanMessage(user_msg)]
    """
    msgs = [SystemMessage(SYSTEM)]
    for m in BLANK:                   # TODO: which sequence replays the earlier turns?
        msgs.append(m)
    msgs.append(HumanMessage(user_msg))
    return msgs

In [ ]:
# --- Self-check: Section 1   (message objects only -- no model call)
h = [HumanMessage("The reference is PMT-1002."), AIMessage("Noted.")]

check("carry() replays every earlier turn",
      lambda: len(carry(h, "which reference?")) == 4)
check("carry() leads with the system message",
      lambda: carry(h, "x")[0].type == "system")
check("carry() preserves the fact from turn 1",
      lambda: any("PMT-1002" in str(m.content) for m in carry(h, "which reference?")),
      "the first turn must survive into the new call")
check("carry() puts the new human message last",
      lambda: carry(h, "which reference?")[-1].content == "which reference?")
check("the turns stay LangChain message objects",
      lambda: all(hasattr(m, "type") for m in carry(h, "x")),
      "append the message objects themselves, not their .content")

## Section 2 &mdash; The loop: tool calls in, tool messages out

Now the real thing. `llm.bind_tools([...])` returns a model that may answer with an **action**
instead of prose. When it does, `response.tool_calls` is a list of
`{"name", "args", "id"}` &mdash; already structured. Your job in the loop is to run the named tool
and hand the result back as a `ToolMessage` carrying the same `id`.

Note what you are **not** doing: parsing "Action: lookup_payment" out of free text. Module 2
shows what that costs when you have to.

In [ ]:
from langchain_core.tools import tool

@tool
def lookup_payment(ref: str) -> str:
    """Return the ledger record for one payment reference such as 'PMT-1002'.

    Use when you need the status, amount, counterparty or reason code of a specific payment.
    Not for searching across payments.
    """
    record = LEDGER.get(ref)
    if record is None:
        return f"no payment found with reference {ref!r}"
    return json.dumps({"ref": ref, **record})


@tool
def policy_for(reason_code: str) -> str:
    """Return the operating policy for one failure reason code, e.g. 'LIMIT_BREACH'.

    Use after you know why a payment failed and need to know what to do about it.
    """
    return POLICY.get(reason_code, f"no policy on file for reason code {reason_code!r}")


TOOLS = {t.name: t for t in (lookup_payment, policy_for)}
print("tools:", list(TOOLS))

In [ ]:
MAX_STEPS = 6

def run_tool_calls(ai_message, tools: dict) -> list:
    """Execute every tool call on an AIMessage. Return the ToolMessages to send back.

    A tool that raises would abort the run, so failures are returned as text the model
    can reason about instead.
    """
    out = []
    for call in ai_message.tool_calls:
        try:
            result = tools[call["name"]].invoke(call["args"])
        except Exception as exc:
            result = f"tool error: {type(exc).__name__}: {exc}"
        out.append(ToolMessage(content=str(result), tool_call_id=BLANK))   # TODO: tie it to the call
    return out


def should_stop(messages: list, steps: int, max_steps: int = MAX_STEPS):
    """Return (stop, reason). The goal is reached when the model answers WITHOUT a tool call."""
    last = messages[-1]
    if getattr(last, "type", None) == "ai" and not last.tool_calls:
        return True, "goal"
    if BLANK:                          # TODO: has the step budget been spent?
        return True, "budget"
    return False, None


def run_agent(question: str, decide, tools: dict, max_steps: int = MAX_STEPS) -> dict:
    """decide(messages) -> AIMessage, possibly carrying tool_calls. The loop is the agent."""
    messages = [SystemMessage(SYSTEM), HumanMessage(question)]
    steps = 0
    while True:
        ai = decide(messages)
        messages.append(ai)
        stop, why = should_stop(messages, steps, max_steps)
        if stop:
            return {"messages": messages, "steps": steps, "stopped": why}
        messages.extend(run_tool_calls(ai, tools))
        steps += 1

In [ ]:
# --- Self-check: Section 2   (a scripted `decide` returns real AIMessages -- no model involved)
def _call(name, args, cid):
    return AIMessage(content="", tool_calls=[{"name": name, "args": args, "id": cid, "type": "tool_call"}])

def _finisher(messages):
    if sum(1 for m in messages if m.type == "tool") >= 2:
        return AIMessage("PMT-1002 failed: INSUFFICIENT_FUNDS. Retry once after 24h.")
    if not any(m.type == "tool" for m in messages):
        return _call("lookup_payment", {"ref": "PMT-1002"}, "c1")
    return _call("policy_for", {"reason_code": "INSUFFICIENT_FUNDS"}, "c2")

def _never_finishes(messages):
    return _call("lookup_payment", {"ref": "PMT-1002"}, f"c{len(messages)}")

check("a ToolMessage is produced per tool call",
      lambda: len(run_tool_calls(_call("lookup_payment", {"ref": "PMT-1002"}, "c1"), TOOLS)) == 1)
check("the ToolMessage carries the call's id",
      lambda: run_tool_calls(_call("lookup_payment", {"ref": "PMT-1002"}, "c9"), TOOLS)[0].tool_call_id == "c9",
      "tool_call_id must be call['id'] -- the model pairs result to request by that id")
check("the ToolMessage carries the tool's real output",
      lambda: "INSUFFICIENT_FUNDS" in run_tool_calls(
          _call("lookup_payment", {"ref": "PMT-1002"}, "c1"), TOOLS)[0].content)
check("an unknown reference does not raise",
      lambda: "no payment found" in run_tool_calls(
          _call("lookup_payment", {"ref": "PMT-9999"}, "c1"), TOOLS)[0].content,
      "a raising tool aborts the whole agent run")
check("a run that answers without a tool call stops with reason 'goal'",
      lambda: run_agent("q", _finisher, TOOLS)["stopped"] == "goal")
check("a run that never finishes stops on the budget",
      lambda: run_agent("q", _never_finishes, TOOLS)["stopped"] == "budget",
      "should_stop() must compare steps against max_steps")
check("the budget is actually respected",
      lambda: run_agent("q", _never_finishes, TOOLS)["steps"] == MAX_STEPS)

## Section 3 &mdash; Loop detection

A budget stops a runaway agent *eventually*. Loop detection stops it **as soon as it stops
learning** &mdash; the same tool, the same arguments, no new information. In production this is
usually the difference between a cheap failure and an expensive one.

Because `tool_calls` is structured, you can detect this exactly, without any string matching.

In [ ]:
def is_looping(messages: list, window: int = 3) -> bool:
    """True when the last `window` tool calls are identical in both name and arguments."""
    calls = [(c["name"], json.dumps(c["args"], sort_keys=True))
             for m in messages if getattr(m, "type", None) == "ai"
             for c in (m.tool_calls or [])]
    if len(calls) < window:
        return False
    return BLANK                      # TODO: are the last `window` calls all the same call?

In [ ]:
# --- Self-check: Section 3
_same  = [_call("lookup_payment", {"ref": "PMT-1002"}, f"c{i}") for i in range(3)]
_mixed = [_call("lookup_payment", {"ref": "PMT-1002"}, "c1"),
          _call("lookup_payment", {"ref": "PMT-1003"}, "c2"),
          _call("lookup_payment", {"ref": "PMT-1002"}, "c3")]

check("three identical calls count as a loop", lambda: is_looping(_same) is True)
check("varied calls are not a loop", lambda: is_looping(_mixed) is False,
      "different arguments mean the agent is still learning something")
check("too short a trace is not yet a loop", lambda: is_looping(_same[:2]) is False)
check("the window is honoured", lambda: is_looping(_same, window=2) is True)
check("the id is ignored -- only name and args identify a call",
      lambda: is_looping(_same) is True,
      "each of those has a different id but the same call")

## Run it for real &mdash; part 1: statelessness

Two calls, where the second depends on the first. Watch the model fail to recall &mdash; then watch
`carry()` fix it, by resending what it already told you.

In [ ]:
if llm_ready():
    llm = get_llm()
    print("--- without carry() -------------------------------------------")
    print("call 1:", llm.invoke([HumanMessage("Remember this reference: PMT-1002. Reply with just OK.")]).content)
    print("call 2:", llm.invoke([HumanMessage("Which payment reference did I just give you?")]).content[:160])

    print("\n--- with carry() ----------------------------------------------")
    def _carried():
        history = [HumanMessage("Remember this reference: PMT-1002."), AIMessage("OK")]
        reply = llm.invoke(carry(history, "Which payment reference did I just give you?"))
        print("call 2:", reply.content[:160])
    guard(_carried)

## Run it for real &mdash; part 2: your loop, driving a real model

`decide` is now the model itself, bound to your two tools. Everything else is the loop you wrote.
Watch the trace: the model asks for the ledger record, you answer, it asks for the policy, you
answer, and only then does it stop.

In [ ]:
if llm_ready():
    def _live_loop():
        model = get_llm().bind_tools(list(TOOLS.values()))
        result = run_agent("Why is PMT-1003 held, and what must we do about it?",
                           lambda msgs: model.invoke(msgs), TOOLS)
        show_messages(result["messages"])
        print(f"\nstopped: {result['stopped']}   steps: {result['steps']}")
        print("looping:", is_looping(result["messages"]))
    guard(_live_loop)

## Run it for real &mdash; part 3: the same agent, in one line

Everything you just wrote &mdash; the loop, the tool dispatch, the `ToolMessage` plumbing, the stop
condition &mdash; is what `create_agent` gives you. Add a **checkpointer** and a `thread_id` and the
message history is kept for you too, which is the whole of Section 1 handled.

Run it and ask a follow-up question that only makes sense if the agent remembered.

In [ ]:
if llm_ready():
    from langchain.agents import create_agent
    from langgraph.checkpoint.memory import InMemorySaver

    agent = create_agent(model=get_llm(), tools=list(TOOLS.values()), system_prompt=SYSTEM,
                         checkpointer=InMemorySaver())
    cfg = {"configurable": {"thread_id": "case-1003"}}

    first = agent.invoke({"messages": [HumanMessage("Why is PMT-1003 held?")]}, cfg)
    show_messages(first["messages"])

    follow = agent.invoke({"messages": [HumanMessage("What was the amount again?")]}, cfg)
    print("\nfollow-up:", follow["messages"][-1].content[:200])
    print(f"messages on this thread: {len(follow['messages'])}")

### Read it

Three things to take away, in order of how much they will cost you later.

1. **You own the state.** `carry()` resends the entire history on *every* turn &mdash; the model
   keeps nothing. Everything an agent "remembers" is something your code chose to put back in
   front of it, which is why Module 3 is about deciding what to keep.
2. **Structured beats parsed.** The model chose its tools through `tool_calls`, so there was no
   format to get wrong. Module 2 shows the same loop without that guarantee.
3. **`create_agent` is the loop you wrote.** Not a different thing &mdash; the same thing, with the
   stop condition, dispatch and history handled. You now know what it is hiding, which is the
   only safe way to use it.

In [ ]:
score()

## Your turn

1. `run_agent` ignores `is_looping`. Wire it in as a third stop reason and give it its own
   label in the return value. Which of the four stop conditions from the slides does that
   still leave unimplemented?
2. Give `create_agent` a **third** tool that overlaps with `lookup_payment` &mdash; say
   `get_payment_status(ref)` &mdash; and see which one it picks. Lab 1.3 turns that into a measurement.
3. Run part 3 again with a second `thread_id`, asking the follow-up question first. Confirm for
   yourself that threads do not leak into each other, then look at where that isolation actually
   lives.